# SESSION 4: Data Collection using Pandas (Part 1)
## Assignment Track 3: Data Science Foundations
**Case Study:** Greenwood Heights Housing Society Electricity Billing & Meter Reading System  
**Author:** Atharv Chaudhari (`ds-work`)

---

### Topics Covered:
1. **What is pandas & why Data Science heavily depends on it**
2. **Multi-format data serialization:** CSV, TSV, Excel (`.xlsx`), JSON (`records`)
3. **Delimiter & Separator Handling:** Commas, semicolons, and tabs (`sep=','`, `sep=';'`, `sep='\t'`)
4. **Large-scale file ingestion with Chunking:** Memory-constrained stream processing using `chunksize`
5. **Early Issue Detection & Data Profiling:** `df.info()`, `df.describe(include='all')`, and custom diagnostic reports (`society_issue_report.csv`)


---
## Task 2: Dataset Generation & Multi-Format Serialization

**Objectives:**
- Generate 1,200 monthly electricity readings for a fictional housing society (*Greenwood Heights*) using `np.random.seed(42)`:
  - `flat_id`: 100 flats spanning four blocks: `A-01` to `D-25`.
  - `month`: 12 billing cycles from `2025-01` to `2025-12`.
  - `units_kwh`: Discrete uniform consumption between 60 and 420 kWh.
  - `amount_inr`: Electricity tariff at ₹7.5 per kWh (`units_kwh * 7.5`).
  - `paid`: Payment collection status (90% 'Yes', 10% 'No').
- Export the primary dataset into 5 standardized data formats with `index=False`:
  1. `readings_comma.csv` (Standard comma-delimited CSV)
  2. `readings_semicolon.csv` (Semicolon-delimited CSV, `sep=';'`)
  3. `readings_tab.tsv` (Tab-delimited TSV, `sep='\t'`)
  4. `readings.xlsx` (Excel workbook via `openpyxl`)
  5. `readings.json` (JSON record array, `orient='records'`)


In [1]:
import numpy as np
import pandas as pd
import json

# Set seed for exact reproducibility
np.random.seed(42)

# 1. Generate Housing Society Flats: 4 Blocks (A, B, C, D) x 25 Flats = 100 Flats
blocks = ['A', 'B', 'C', 'D']
flats = [f"{b}-{i:02d}" for b in blocks for i in range(1, 26)]
months = [f"2025-{m:02d}" for m in range(1, 13)]

# Build full Cartesian product: 100 flats x 12 months = 1,200 monthly readings
records = [{'flat_id': f, 'month': m} for m in months for f in flats]
df_readings = pd.DataFrame(records)

# 2. Generate consumption (60 to 420 kWh), billing tariff, and payment status
df_readings['units_kwh'] = np.random.randint(60, 421, size=len(df_readings))
df_readings['amount_inr'] = df_readings['units_kwh'] * 7.5
df_readings['paid'] = np.random.choice(['Yes', 'No'], size=len(df_readings), p=[0.9, 0.1])

print(f"Generated Dataset Shape: {df_readings.shape} (1,200 readings x 5 columns)")
print("\nFirst 5 Records:")
print(df_readings.head())
print("\nPayment Status Distribution:")
print(df_readings['paid'].value_counts(normalize=True))


Generated Dataset Shape: (1200, 5) (1,200 readings x 5 columns)

First 5 Records:
  flat_id    month  units_kwh  amount_inr paid
0    A-01  2025-01        162      1215.0  Yes
1    A-02  2025-01        408      3060.0  Yes
2    A-03  2025-01        330      2475.0  Yes
3    A-04  2025-01        166      1245.0  Yes
4    A-05  2025-01        131       982.5  Yes

Payment Status Distribution:
paid
Yes    0.906667
No     0.093333
Name: proportion, dtype: float64


In [2]:
# 3. Export to 5 distinct data interchange formats (index=False)
df_readings.to_csv('readings_comma.csv', index=False)
df_readings.to_csv('readings_semicolon.csv', sep=';', index=False)
df_readings.to_csv('readings_tab.tsv', sep='\t', index=False)
df_readings.to_excel('readings.xlsx', index=False)
df_readings.to_json('readings.json', orient='records', indent=2)

print("Successfully exported dataset to 5 formats:")
print(" - readings_comma.csv     (Standard CSV)")
print(" - readings_semicolon.csv (sep=';')")
print(" - readings_tab.tsv       (sep='\\t')")
print(" - readings.xlsx          (Excel via openpyxl)")
print(" - readings.json          (orient='records')")


Successfully exported dataset to 5 formats:
 - readings_comma.csv     (Standard CSV)
 - readings_semicolon.csv (sep=';')
 - readings_tab.tsv       (sep='\t')
 - readings.xlsx          (Excel via openpyxl)
 - readings.json          (orient='records')


---
## Task 3: Multi-Format Ingestion & Integrity Verification

**Objectives:**
- Ingest all 5 generated files back into separate pandas DataFrames using:
  - `pd.read_csv('readings_comma.csv')`
  - `pd.read_csv('readings_semicolon.csv', sep=';')`
  - `pd.read_csv('readings_tab.tsv', sep='\t')`
  - `pd.read_excel('readings.xlsx')`
  - `pd.read_json('readings.json')`
- Produce a comparative diagnostic audit table displaying:
  - File Name
  - Ingested Shape
  - Total `units_kwh` consumption sum
- Provide a programmatic assertion verifying 100% data integrity across all file formats.


In [3]:
# Load all 5 files back using respective parsers and separators
df_csv = pd.read_csv('readings_comma.csv')
df_semi = pd.read_csv('readings_semicolon.csv', sep=';')
df_tsv = pd.read_csv('readings_tab.tsv', sep='\t')
df_excel = pd.read_excel('readings.xlsx')
df_json = pd.read_json('readings.json')

files_audit = [
    ('readings_comma.csv', df_csv),
    ('readings_semicolon.csv', df_semi),
    ('readings_tab.tsv', df_tsv),
    ('readings.xlsx', df_excel),
    ('readings.json', df_json)
]

print("=" * 65)
print(f"{'File Name':<25} | {'Shape':<12} | {'units_kwh Sum':<18}")
print("=" * 65)

kwh_sums = []
shapes = []
for fname, df_i in files_audit:
    s = df_i['units_kwh'].sum()
    kwh_sums.append(s)
    shapes.append(df_i.shape)
    print(f"{fname:<25} | {str(df_i.shape):<12} | {s:<18,}")
print("=" * 65)

# Verification check
all_shapes_match = len(set(shapes)) == 1
all_sums_match = len(set(kwh_sums)) == 1
all_five_match = all_shapes_match and all_sums_match

print(f"\nFinal Verification: All five files match identically in shape and units_kwh sum: {all_five_match}")


File Name                 | Shape        | units_kwh Sum     
readings_comma.csv        | (1200, 5)    | 289,085           
readings_semicolon.csv    | (1200, 5)    | 289,085           
readings_tab.tsv          | (1200, 5)    | 289,085           
readings.xlsx             | (1200, 5)    | 289,085           
readings.json             | (1200, 5)    | 289,085           

Final Verification: All five files match identically in shape and units_kwh sum: True


---
## Task 4: Chunked Ingestion for Large Data Files (`chunksize`)

**Objectives:**
- Generate `meter_big.csv` containing 60,000 hourly smart-meter readings:
  - `flat_id`: Sampled from housing society flats.
  - `hour_no`: Hourly timestamps (1 through 24 repeated).
  - `kwh`: Realistic power draws between 0.1 and 2.5 kWh, rounded to 2 decimal places.
- Ingest `meter_big.csv` using chunked streaming: `pd.read_csv('meter_big.csv', chunksize=10000)`.
- Loop over the file chunks to aggregate total row counts and cumulative energy consumption without ever loading the full dataset into RAM.
- Verify chunk count (6) and confirm row count and total kWh match the generating DataFrame within 0.01 kWh.
- *Hint applied:* Each chunk yielded by `TextFileReader` is an individual DataFrame; use `chunk['kwh'].sum()` inside the loop.


In [4]:
# 1. Generate 60,000 hourly smart meter readings
np.random.seed(42)

df_meter_big = pd.DataFrame({
    'flat_id': np.random.choice(flats, size=60000),
    'hour_no': np.tile(np.arange(1, 25), 2500),
    'kwh': np.round(np.random.uniform(0.1, 2.5, size=60000), 2)
})

# Save to CSV
df_meter_big.to_csv('meter_big.csv', index=False)

# Ground truth values from source DataFrame
gt_rows = len(df_meter_big)
gt_kwh = df_meter_big['kwh'].sum()

print(f"Generated meter_big.csv: {gt_rows:,} rows | Ground Truth Total kWh: {gt_kwh:,.2f}")


Generated meter_big.csv: 60,000 rows | Ground Truth Total kWh: 78,216.07


In [5]:
# 2. Chunked ingestion: Stream file in batches of 10,000 rows
chunk_counter = 0
total_chunk_rows = 0
total_chunk_kwh = 0.0

chunk_iterator = pd.read_csv('meter_big.csv', chunksize=10000)

print("Streaming meter_big.csv in chunks of 10,000 rows:")
for chunk in chunk_iterator:
    chunk_counter += 1
    c_rows = len(chunk)
    c_kwh = chunk['kwh'].sum()
    total_chunk_rows += c_rows
    total_chunk_kwh += c_kwh
    print(f"  Chunk {chunk_counter}: Processed {c_rows:,} rows | Subtotal kWh: {c_kwh:,.2f}")

print("\n=== Chunk Processing Results vs Ground Truth ===")
print(f"Total Chunks Processed : {chunk_counter} (Expected: 6)")
print(f"Total Rows Processed   : {total_chunk_rows:,} (Ground Truth: {gt_rows:,})")
print(f"Total kWh Accumulated  : {total_chunk_kwh:,.2f} (Ground Truth: {gt_kwh:,.2f})")

# Tolerance assertion: difference must be within 0.01
kwh_diff = abs(total_chunk_kwh - gt_kwh)
assert chunk_counter == 6, "Chunk count must be 6!"
assert total_chunk_rows == gt_rows, "Total rows must match ground truth!"
assert kwh_diff <= 0.01, f"Total kWh difference {kwh_diff} exceeds 0.01 tolerance!"
print(f"Assertion PASSED: Difference {kwh_diff:.6f} is well within the 0.01 tolerance limit!")


Streaming meter_big.csv in chunks of 10,000 rows:
  Chunk 1: Processed 10,000 rows | Subtotal kWh: 13,096.05
  Chunk 2: Processed 10,000 rows | Subtotal kWh: 13,108.24
  Chunk 3: Processed 10,000 rows | Subtotal kWh: 13,053.41
  Chunk 4: Processed 10,000 rows | Subtotal kWh: 12,946.77
  Chunk 5: Processed 10,000 rows | Subtotal kWh: 13,043.24
  Chunk 6: Processed 10,000 rows | Subtotal kWh: 12,968.36

=== Chunk Processing Results vs Ground Truth ===
Total Chunks Processed : 6 (Expected: 6)
Total Rows Processed   : 60,000 (Ground Truth: 60,000)
Total kWh Accumulated  : 78,216.07 (Ground Truth: 78,216.07)
Assertion PASSED: Difference 0.000000 is well within the 0.01 tolerance limit!


---
## Task 1: Realistic Messy Dataset Creation & Early Issue Detection

**Objectives:**
- Construct `readings_messy.csv` from the 1,200 readings by deliberately introducing 3 common data defects:
  1. **Missing Data:** Blank 5% of entries in `units_kwh` (60 missing values).
  2. **Type Contamination:** Format `amount_inr` into text strings such as `'Rs 1234.50'`.
  3. **Categorical Inconsistency:** Mix letter casing for `paid` (`'yes'`, `'YES'`, `'No'`, `'no'`, `'Yes'`).
  4. Save as `readings_messy.csv` (`index=False`).
- Load `readings_messy.csv` into a clean DataFrame.
- Inspect the schema using `df.info()` and statistical distributions using `df.describe(include='all')`.
- Build a dedicated audit DataFrame `issue_report` with one row per column capturing:
  - `column`: Field name
  - `dtype`: Inferred data type
  - `missing_count`: Count of null / NaN values
  - `unique_count`: Cardinality of unique entries
- Save audit report as `society_issue_report.csv` (`index=False`).
- *Constraint applied:* **Only detect the problems; do not fix them yet.**


In [6]:
# 1. Inject real-world data flaws into a fresh copy
np.random.seed(42)
df_messy = df_readings.copy()

# Defect A: Blank 5% of units_kwh (60 rows out of 1,200)
n_blank = int(len(df_messy) * 0.05)
blank_indices = np.random.choice(df_messy.index, size=n_blank, replace=False)
df_messy.loc[blank_indices, 'units_kwh'] = np.nan

# Defect B: Convert numeric amount_inr into text currency strings
df_messy['amount_inr'] = df_messy['amount_inr'].apply(lambda x: f"Rs {x:.2f}")

# Defect C: Inconsistent casing in paid column
mixed_cases = ['yes', 'YES', 'No', 'no', 'Yes']
df_messy['paid'] = np.random.choice(mixed_cases, size=len(df_messy), p=[0.35, 0.35, 0.1, 0.1, 0.1])

# Save messy dataset
df_messy.to_csv('readings_messy.csv', index=False)
print("Created readings_messy.csv with injected data anomalies.")


Created readings_messy.csv with injected data anomalies.


In [7]:
# 2. Ingest messy dataset and perform early issue detection
df_loaded_messy = pd.read_csv('readings_messy.csv')

print("=== df.info() Diagnostic ===")
df_loaded_messy.info()

print("\n=== df.describe(include='all') Summary ===")
print(df_loaded_messy.describe(include='all'))


=== df.info() Diagnostic ===
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   flat_id     1200 non-null   str    
 1   month       1200 non-null   str    
 2   units_kwh   1140 non-null   float64
 3   amount_inr  1200 non-null   str    
 4   paid        1200 non-null   str    
dtypes: float64(1), str(4)
memory usage: 47.0 KB

=== df.describe(include='all') Summary ===
       flat_id    month    units_kwh  amount_inr  paid
count     1200     1200  1140.000000        1200  1200
unique     100       12          NaN         344     5
top       A-01  2025-01          NaN  Rs 3037.50   YES
freq        12      100          NaN           9   422
mean       NaN      NaN   240.893860         NaN   NaN
std        NaN      NaN   103.079724         NaN   NaN
min        NaN      NaN    60.000000         NaN   NaN
25%        NaN      NaN   157.750000         NaN   NaN
5

In [8]:
# 3. Construct issue_report table
report_rows = []
for col in df_loaded_messy.columns:
    report_rows.append({
        'column': col,
        'dtype': str(df_loaded_messy[col].dtype),
        'missing_count': int(df_loaded_messy[col].isna().sum()),
        'unique_count': int(df_loaded_messy[col].nunique())
    })

issue_report = pd.DataFrame(report_rows)

# Save issue report to CSV
issue_report.to_csv('society_issue_report.csv', index=False)

print("=== Society Issue Report (society_issue_report.csv) ===")
print(issue_report.to_string(index=False))

print("\nKey Anomalies Flagged:")
print(" 1. units_kwh has 60 missing values (5.0% null rate).")
print(" 2. amount_inr has dtype 'object' (string) due to 'Rs' prefix, preventing mathematical aggregations.")
print(" 3. paid column contains 5 unique categories ('yes', 'YES', 'No', 'no', 'Yes') instead of binary categories.")


=== Society Issue Report (society_issue_report.csv) ===
    column   dtype  missing_count  unique_count
   flat_id     str              0           100
     month     str              0            12
 units_kwh float64             60           339
amount_inr     str              0           344
      paid     str              0             5

Key Anomalies Flagged:
 1. units_kwh has 60 missing values (5.0% null rate).
 2. amount_inr has dtype 'object' (string) due to 'Rs' prefix, preventing mathematical aggregations.
 3. paid column contains 5 unique categories ('yes', 'YES', 'No', 'no', 'Yes') instead of binary categories.


---
## Summary of Key Takeaways:

1. **Why Pandas is the Standard for Data Ingestion:**
   - Pandas delivers flexible multi-format parsers (`read_csv`, `read_excel`, `read_json`) with built-in parameter customization (`sep`, `orient`, `chunksize`).
2. **Separator Awareness:**
   - Never assume CSV files use commas; international and enterprise exports frequently utilize semicolons (`;`) or tabs (`\t`). Always inspect delimiters.
3. **Out-of-Core Processing with `chunksize`:**
   - Datasets that exceed available system memory can be processed sequentially in chunks using iterator streams, accumulating metrics in $O(1)$ memory.
4. **Early Detection Saves Pipeline Downstream Failure:**
   - Running `df.info()` and `df.describe(include='all')` immediately flags data corruption, string contamination in financial fields, and high-cardinality boolean columns before feeding models.
